[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_03/15_guias_de_onda_y_antenas.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 15 — Guías de onda rectangulares y antenas

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 3**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Calcular la frecuencia de corte de los modos de una guía rectangular.
2. Determinar qué modos se propagan a una frecuencia dada y cuáles no.
3. Calcular $\lambda_g$, la velocidad de fase y la velocidad de grupo.
4. Evaluar la eficiencia y la ganancia de una antena corta.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "guias_y_antenas.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import VELOCIDAD_LUZ
from guias_y_antenas import (
    frecuencia_de_corte,
    longitud_onda_guia,
    velocidad_fase_guia,
    velocidad_grupo_guia,
    resistencia_radiacion_dipolo_corto,
    eficiencia_antena,
    ganancia_antena,
    a_decibelios,
    potencia_radiada_dipolo,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Por qué una guía tiene frecuencia de corte

Dentro de una guía metálica hueca, el campo debe anularse en las paredes. Esa
condición solo la cumplen ciertos patrones transversales, llamados **modos**.

Cada modo necesita que quepa al menos media longitud de onda en la dimensión
correspondiente. Si la frecuencia es demasiado baja, la longitud de onda es
demasiado grande y no cabe: el modo no se propaga, se apaga
exponencialmente. Ésa es la frecuencia de corte.

Una guía es, en el fondo, un filtro pasaaltos.

### 2.2 Por qué interesa el rango monomodo

Entre el corte del primer modo y el del segundo, solo uno se propaga. En ese
rango la señal viaja de forma limpia y predecible.

Si se opera más arriba, varios modos coexisten, cada uno con su propia
velocidad, y la señal se distorsiona. Por eso las guías comerciales se
especifican con un rango de frecuencias recomendado.

### 2.3 Antenas: radiar no es disipar

En una antena hay dos resistencias distintas. La de pérdidas, $R_p$,
representa el calor que se pierde en el conductor. La de radiación, $R_r$,
representa la potencia que sale al espacio.

Las dos consumen potencia de la fuente, pero solo una hace algo útil. La
eficiencia es la fracción que se va por la buena.

El problema de las antenas cortas es que $R_r$ va con $(dl/\lambda)^2$: si la
antena es diez veces más corta, radia cien veces peor.

## 3. Ecuaciones

**Frecuencia de corte del modo TE$_{mn}$ o TM$_{mn}$** en una guía de
dimensiones $a \times b$:

$$
f_{c,mn} = \frac{c}{2}\sqrt{\left(\frac{m}{a}\right)^{2}
                         + \left(\frac{n}{b}\right)^{2}} .
$$

El modo dominante en una guía con $a > b$ es el TE$_{10}$, con
$f_c = c/(2a)$.

**Propagación por encima del corte:**

$$
\lambda_g = \frac{\lambda_0}{\sqrt{1 - (f_c/f)^{2}}},
\qquad
u_p = \frac{c}{\sqrt{1 - (f_c/f)^{2}}},
\qquad
u_g = c\sqrt{1 - (f_c/f)^{2}} .
$$

De aquí sale una relación notable:

$$
u_p\,u_g = c^{2}.
$$

**Antena corta:**

$$
R_r = 80\pi^{2}\left(\frac{dl}{\lambda}\right)^{2},
\qquad
\xi = \frac{R_r}{R_r + R_p},
\qquad
G = \xi D,
\qquad
P_{\text{rad}} = \tfrac{1}{2}I_0^{2}R_r .
$$

Para un dipolo hertziano la directividad es $D = 1.5$.

## 4. Qué significa físicamente

**La velocidad de fase supera a $c$, y no pasa nada.** $u_p > c$ solo dice a
qué velocidad se desplazan los frentes de fase, que es un patrón geométrico,
no información. Lo que transporta energía es la velocidad de grupo, y ésa
siempre es menor que $c$. El producto de ambas da $c^2$ exactamente.

**$\lambda_g$ es mayor que en el espacio libre.** La onda rebota en zigzag
entre las paredes, así que avanza menos por cada ciclo. Esto importa al
diseñar: un tramo de "un cuarto de onda" dentro de una guía es más largo de
lo que uno esperaría.

**La eficiencia de una antena corta es mala.** Con $dl/\lambda = 0.05$ resulta
$R_r \approx 2~\Omega$. Si la resistencia de pérdidas es de 1 $\Omega$, se
pierde un tercio de la potencia en calor. Ése es el problema de fondo de las
antenas pequeñas de los teléfonos.

**Ganancia negativa en dBi es posible.** Aunque la directividad sea 1.5
(positiva en dB), la baja eficiencia puede llevar la ganancia por debajo de 1,
es decir, a dBi negativos. La antena concentra la señal, pero pierde tanta
potencia que termina peor que un radiador isotrópico ideal.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: guía rectangular WR-90 ---
ancho = 22.86e-3     # dimensión mayor a [m]
alto = 10.16e-3      # dimensión menor b [m]
frecuencia = 10.0e9  # frecuencia de operación [Hz]

# --- Problema 2: antena corta ---
dl_sobre_lambda = 0.05      # longitud del dipolo en longitudes de onda
resistencia_perdidas = 1.0  # resistencia de pérdidas R_p [ohm]
corriente_pico = 0.1        # corriente pico I_0 [A]
directividad = 1.5          # directividad del dipolo hertziano

## 6. Implementación

### 6.1 Problema 1 — modos de la guía

In [ ]:
MODOS = {"TE10": (1, 0), "TE20": (2, 0), "TE01": (0, 1), "TM11": (1, 1)}

cortes = {
    nombre: frecuencia_de_corte(ancho, alto, m, n)
    for nombre, (m, n) in MODOS.items()
}
corte_dominante = cortes["TE10"]

propagantes = [nombre for nombre, fc in cortes.items() if frecuencia > fc]

lambda_guia = longitud_onda_guia(frecuencia, corte_dominante)
velocidad_fase = velocidad_fase_guia(frecuencia, corte_dominante)
velocidad_grupo = velocidad_grupo_guia(frecuencia, corte_dominante)

### 6.2 Problema 2 — la antena

In [ ]:
R_radiacion = resistencia_radiacion_dipolo_corto(dl_sobre_lambda)
eficiencia = eficiencia_antena(R_radiacion, resistencia_perdidas)
ganancia = ganancia_antena(eficiencia, directividad)
ganancia_dbi = a_decibelios(ganancia)
potencia_radiada = potencia_radiada_dipolo(corriente_pico, R_radiacion)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [(f"Corte del modo {nombre}", "f_c", fc / 1.0e9, "GHz")
     for nombre, fc in cortes.items()]
    + [
        ("Frecuencia de operación", "f", frecuencia / 1.0e9, "GHz"),
        ("Longitud de onda en la guía", "lambda_g", lambda_guia * 1000.0, "mm"),
        ("Longitud de onda libre", "lambda_0", VELOCIDAD_LUZ / frecuencia * 1000.0, "mm"),
        ("Velocidad de fase", "u_p", velocidad_fase, "m/s"),
        ("Velocidad de grupo", "u_g", velocidad_grupo, "m/s"),
        ("Producto normalizado", "u_p u_g / c^2", velocidad_fase * velocidad_grupo / VELOCIDAD_LUZ**2, "-"),
    ]
)

In [ ]:
print("Modos que se propagan a esta frecuencia:", ", ".join(propagantes))
print(f"Número de modos propagantes: {len(propagantes)}")
if len(propagantes) == 1:
    print("La guía trabaja en régimen monomodo: es el rango recomendado.")
else:
    print("Hay más de un modo: la señal puede distorsionarse.")

In [ ]:
tabla_resultados(
    [
        ("Resistencia de radiación", "R_r", R_radiacion, "ohm"),
        ("Resistencia de pérdidas", "R_p", resistencia_perdidas, "ohm"),
        ("Eficiencia", "xi", eficiencia, "-"),
        ("Eficiencia", "xi", eficiencia * 100.0, "%"),
        ("Ganancia lineal", "G", ganancia, "-"),
        ("Ganancia en decibelios", "G", ganancia_dbi, "dBi"),
        ("Potencia radiada", "P_rad", potencia_radiada, "W"),
    ]
)

## 8. Visualización

A la izquierda, las frecuencias de corte comparadas con la de operación. A la
derecha, el patrón de radiación del dipolo con su ancho de haz.

In [ ]:
fig, (eje_modos, eje_patron) = plt.subplots(1, 2, figsize=(9.5, 4.0))

nombres = list(cortes)
valores = [cortes[nombre] / 1.0e9 for nombre in nombres]
colores = ["tab:green" if frecuencia > cortes[n] else "tab:gray" for n in nombres]

eje_modos.bar(nombres, valores, color=colores)
eje_modos.axhline(frecuencia / 1.0e9, color="tab:red", linestyle="--",
                  label="frecuencia de operación")
eje_modos.set_ylabel("Frecuencia de corte (GHz)")
eje_modos.set_title("Verde: se propaga. Gris: no.")
eje_modos.legend(fontsize=8)

theta = np.linspace(0.0, np.pi, 400)
eje_patron.plot(np.rad2deg(theta), np.sin(theta) ** 2)
eje_patron.axhline(0.5, color="black", linestyle="--", label="mitad de potencia")
eje_patron.axvline(45.0, color="gray", linestyle=":", linewidth=0.8)
eje_patron.axvline(135.0, color="gray", linestyle=":", linewidth=0.8)
eje_patron.set_xlabel("theta (grados)")
eje_patron.set_ylabel("Potencia normalizada")
eje_patron.set_title("Patrón del dipolo: ancho de haz de 90 grados")
eje_patron.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**A 10 GHz la WR-90 es monomodo.** Solo el TE$_{10}$ tiene su corte por
debajo, en 6.56 GHz; el siguiente está en 13.1 GHz. El código lo confirma
explícitamente. Por eso la banda recomendada de esta guía es
aproximadamente 8.2–12.4 GHz.

**$\lambda_g$ supera a $\lambda_0$ en un 32 %.** En el espacio libre a 10 GHz
la longitud de onda es de 30 mm; dentro de la guía es de casi 40 mm. Si usted
diseñara un tramo de $\lambda/4$ usando el valor libre, se equivocaría por
2.5 mm.

**$u_p u_g = c^2$ hasta la última cifra.** La tabla muestra el producto
normalizado igual a 1.0 exacto. La velocidad de fase supera a $c$ en un 32 %
y la de grupo se queda un 25 % por debajo, y el producto se compensa.

**La antena tiene ganancia negativa en dBi.** Con $R_r = 1.97~\Omega$ frente a
1 $\Omega$ de pérdidas, la eficiencia es del 66 % y la ganancia resulta
ligeramente menor que 1, es decir, unos $-0.02$ dBi. Radia menos que un
radiador isotrópico ideal, aunque concentre la energía en el ecuador.

**El ancho de haz es de 90 grados.** Las líneas punteadas en 45° y 135°
marcan dónde la potencia cae a la mitad. Un dipolo corto es una antena muy
poco direccional.

## 10. Ejercicios para experimentar

            1. Baje `frecuencia` a `5.0e9`. ¿Qué modos se propagan? Lea el mensaje del
               código: ¿qué significa que no se propague ninguno?
            2. Suba `frecuencia` a `14.0e9`. ¿Cuántos modos hay ahora? ¿Por qué eso es un
               problema?
            3. Duplique `ancho`. ¿Qué le pasa al corte del TE$_{10}$? ¿Y al rango
               monomodo?
            4. Duplique `dl_sobre_lambda` a `0.1`. ¿Por qué factor crece $R_r$? Verifique
               que va con el cuadrado.
            5. Baje `resistencia_perdidas` a `0.01` $\Omega$. ¿Cuánto sube la eficiencia?
               ¿Cuánto vale ahora la ganancia en dBi?
            6. ¿Qué valor de `dl_sobre_lambda` hace falta para llegar al 90 % de
               eficiencia con $R_p = 1~\Omega$? Despeje a mano y compruebe aquí.